<a href="https://colab.research.google.com/github/Mimiuex/save-kiwis-from-flooding/blob/hyunbin/notebooks/flood_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flood Mapping Data Pipeline

GEOG761 Group 05 — Flood Mapping & Response

Fetches the six input layers from the project's architecture diagram for the
**Northland region** and saves them in `data/raw/`:

| Layer | Dataset | Provider | Access | Resolution | Date | CRS | Format |
|---|---|---|---|---|---|---|---|
| Land Type | ESA WorldCover v200 | ESA | Earth Engine | 10 m | 2021 | EPSG:4326 | GeoTIFF |
| Elevation | NZ LiDAR 1 m DEM (LDS layer 121859) | LINZ | Public S3 bucket, no key | 10 m (resampled from 1 m) | National surveys 2008–2026; LDS layer updated 18 Sep 2026 | EPSG:4326 (warped from NZTM) | GeoTIFF |
| SAR Satellite | Sentinel-1 GRD, IW mode, VV + VH | ESA Copernicus | Earth Engine | 10 m | 14 Feb 2023, Cyclone Gabrielle | EPSG:4326 | GeoTIFF |
| Population Density | WorldPop Global Project 100 m | WorldPop | Earth Engine | 100 m | 2020 estimate (latest year WorldPop has for NZ) | EPSG:4326 | GeoTIFF |
| Critical Infrastructure | NZ Building Outlines (LDS layer 101290) | LINZ | LDS WFS, API key | Vector | Footprints traced from 2014–2016 aerial photos in Northland (new buildings missing); LDS layer updated 18 May 2026 | EPSG:4326 | GeoJSON |
| Transport Infrastructure | NZ Road Centrelines, Topo 1:50k (LDS layer 50329) | LINZ | LDS WFS, API key | Vector | LDS layer updated 30 Sep 2026 (Topo50 map data; roads carry no individual date) | EPSG:4326 | GeoJSON |

Run the cells top to bottom. The Earth Engine cell below opens a one-time
browser login the first time you run this notebook in a session. The
building and road layers (LINZ WFS) need a LINZ API key stored as a Colab
secret (see section 4); the DEM doesn't.

Outputs go to Google Drive by default so they survive a Colab runtime reset
— see section 3.


## 1. Setup

In [1]:
# Runtime packages (earthengine-api and requests ship with Colab). geedim backs
# `geemap.download_ee_image`, which the `export_ee_image` helper in section 2
# uses to split large exports into tiles under Earth Engine's request limit.
!pip install -q geemap geopandas rasterio geedim

In [2]:
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path
from urllib.parse import urljoin

import ee
import geemap
import geopandas as gpd
import rasterio
import rasterio.shutil
import requests
from google.colab import drive, userdata
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds
from shapely.geometry import box


## 2. Functions

One fetch function per layer, plus the Earth Engine export helper. They read
dataset IDs and the LINZ key from the Configuration cell (section 4), so they
only need to be defined here; nothing runs until sections 5–10 call them.


In [3]:
def export_ee_image(image, path, aoi, scale):
    """Download an Earth Engine image to GeoTIFF in EPSG:4326. `geemap.download_ee_image`
    (backed by geedim) tiles large requests into many small ones, so it works
    at any AOI size."""
    geemap.download_ee_image(image, filename=path, region=aoi, scale=scale, crs="EPSG:4326")


def fetch_land_cover(aoi):
    image = ee.ImageCollection(WORLDCOVER_COLLECTION).first()
    return image.select("Map").clip(aoi).rename("land_type")


def fetch_linz_dem(aoi_poly, out_path, res_m):
    """Mosaic the LINZ LiDAR 1 m DEM tiles covering aoi_poly, read at res_m
    and written in EPSG:4326 (the tiles are NZTM) on a grid starting at the
    AOI's north-west corner."""
    links = requests.get(LINZ_DEM_COLLECTION, timeout=60).json()["links"]
    item_urls = [urljoin(LINZ_DEM_COLLECTION, l["href"]) for l in links if l["rel"] == "item"]
    with ThreadPoolExecutor(16) as pool:  # ~400 small tile-index JSONs, one per Topo50 sheet
        items = list(pool.map(lambda u: (u, requests.get(u, timeout=60).json()), item_urls))
    tiles = [
        "/vsicurl/" + urljoin(url, item["assets"]["visual"]["href"])
        for url, item in items if box(*item["bbox"]).intersects(aoi_poly)
    ]
    if not tiles:
        raise RuntimeError("No LINZ 1 m DEM tile covers this AOI.")
    print(f"  {len(tiles)} DEM tile(s) intersect the AOI")
    west, south, east, north = aoi_poly.bounds
    nztm_path = f"{out_path}.nztm.tmp"  # merge in NZTM (metres) first, then warp to lat/lon
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
        merge(tiles, bounds=transform_bounds("EPSG:4326", "EPSG:2193", west, south, east, north),
              res=res_m, resampling=Resampling.average, dst_path=nztm_path,
              dst_kwds={"driver": "GTiff", "compress": "deflate", "predictor": 3, "tiled": True})
    res_deg = res_m / 111_320  # ~metres per degree of latitude; pixels are narrower than tall at -35 deg
    width, height = round((east - west) / res_deg), round((north - south) / res_deg)
    with rasterio.open(nztm_path) as src, WarpedVRT(
        src, crs="EPSG:4326", transform=from_origin(west, north, res_deg, res_deg),
        width=width, height=height, resampling=Resampling.bilinear, nodata=src.nodata,
    ) as vrt:
        rasterio.shutil.copy(vrt, out_path, driver="GTiff", compress="deflate", predictor=3, tiled=True)
    Path(nztm_path).unlink()
    return out_path


def fetch_sar_collection(aoi, start_date, end_date):
    return (
        ee.ImageCollection(S1_COLLECTION)
        .filterBounds(aoi)
        .filterDate(str(start_date), str(end_date))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
    )


def fetch_population(aoi, year):
    collection = (
        ee.ImageCollection(WORLDPOP_COLLECTION)
        .filter(ee.Filter.eq("country", "NZL"))
        .filter(ee.Filter.eq("year", year))
    )
    return collection.mosaic().clip(aoi).rename("population_density")


def fetch_linz_wfs(layer_id, aoi_poly, page_size=10_000):
    """All features of a LINZ Data Service layer inside aoi_poly."""
    west, south, east, north = aoi_poly.bounds
    params = {
        "service": "WFS", "version": "2.0.0", "request": "GetFeature",
        "typeNames": f"layer-{layer_id}", "outputFormat": "json", "srsName": "EPSG:4326",
        "bbox": f"{south},{west},{north},{east},urn:ogc:def:crs:EPSG::4326",  # lat/lon order
        "count": page_size,
    }
    url = f"https://data.linz.govt.nz/services;key={LINZ_API_KEY}/wfs"
    features = []
    while True:
        r = requests.get(url, params={**params, "startIndex": len(features)}, timeout=300)
        r.raise_for_status()
        page = r.json()["features"]
        features += page
        if len(page) < page_size:
            break
    if not features:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    # WFS filters by bounding box; clip to the actual region outline.
    return gpd.GeoDataFrame.from_features(features, crs="EPSG:4326").clip(aoi_poly)


## 3. Earth Engine login and Google Drive

The Earth Engine cell opens a one-time browser login per Colab session.

With `USE_DRIVE = True` outputs are written under
`/content/drive/MyDrive/save-kiwis-from-flooding/data/raw/` and survive a
runtime reset. Set it to `False` to write to `/content/data/raw/` instead.


In [4]:
# One-time interactive login per Colab session/account. If this cell has
# already been authorised for this account, it returns immediately.
ee.Authenticate()

# Use your group's GEE-registered Cloud project.
EE_PROJECT = "basic-thinker-504003-q7"
ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready, project:", EE_PROJECT)

Earth Engine ready, project: basic-thinker-504003-q7


In [5]:
USE_DRIVE = True  # False: outputs only live in /content until the runtime resets

if USE_DRIVE:
    drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Configuration

In [6]:
# --- Output paths ------------------------------------------------------------
BASE_DIR = Path("/content/drive/MyDrive/GEOG 761 Group Project") if USE_DRIVE else Path("/content")
RAW_DIR = BASE_DIR / "Project Data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# --- Area of interest: Northland region ---------------------------------------
# FAO GAUL level-1 boundaries in Earth Engine include NZ's regions.
# Run once per region for the event notebook: ("northland", "Northland"), then ("hawkes_bay", "Hawke's Bay").
AOI_NAME, ADM1_NAME = "northland", "Northland"
NZ_REGIONS = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq("ADM0_NAME", "New Zealand"))
AOI_FC = NZ_REGIONS.filter(ee.Filter.eq("ADM1_NAME", ADM1_NAME))
if AOI_FC.size().getInfo() == 0:
    raise RuntimeError(
        "Northland not found in FAO/GAUL/2015/level1. NZ region names there: "
        f"{sorted(NZ_REGIONS.aggregate_array('ADM1_NAME').getInfo())}"
    )
AOI = AOI_FC.geometry()  # Earth Engine layers (sections 5, 7, 8)
AOI_POLY = geemap.ee_to_gdf(AOI_FC).to_crs(4326).geometry.union_all()  # LINZ layers (sections 6, 9, 10)

# --- Event window: Cyclone Gabrielle -------------------------------------------
# Gabrielle hit Northland 12-14 Feb 2023 (NZDT). The Sentinel-1A ascending
# pass on 14 Feb 07:08 UTC (20:08 NZDT) covers almost all of Northland in one
# orbit; this window selects only that pass (END_DATE is exclusive).
START_DATE, END_DATE = date(2023, 2, 14), date(2023, 2, 15)

# --- LINZ Data Service ---------------------------------------------------------
# Needed for Building Outlines + Road Centrelines (sections 9-10). Free key:
# data.linz.govt.nz -> profile menu -> "API Keys" -> "Generate API key",
# enabling "Web Feature Service (WFS)". Store it as a Colab secret named
# LINZ_API_KEY (key icon in the left sidebar), not in this notebook.
LINZ_API_KEY = userdata.get("LINZ_API_KEY")

# --- Dataset sources -----------------------------------------------------------
WORLDCOVER_COLLECTION = "ESA/WorldCover/v200"
S1_COLLECTION = "COPERNICUS/S1_GRD"
WORLDPOP_COLLECTION = "WorldPop/GP/100m/pop"
WORLDPOP_YEAR = 2020
LINZ_DEM_COLLECTION = "https://nz-elevation.s3.ap-southeast-2.amazonaws.com/new-zealand/new-zealand/dem_1m/2193/collection.json"

# --- Resolution ------------------------------------------------------------
WORLDCOVER_SCALE_M = 10   # native resolution (section 5)
DEM_RES_M = 10            # LINZ LiDAR DEM (section 6); native 1 m for all of Northland is tens of GB
SAR_SCALE_M = 10          # Sentinel-1 native resolution (section 7)
POPULATION_SCALE_M = 100  # WorldPop native resolution (section 8)

print("AOI:", AOI_NAME, "| bounds:", [round(v, 3) for v in AOI_POLY.bounds])
print("SAR window:", START_DATE, "to", END_DATE)
print("Outputs under:", RAW_DIR)


/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


AOI: northland | bounds: [172.113, -36.397, 174.779, -34.136]
SAR window: 2023-02-14 to 2023-02-15
Outputs under: /content/drive/MyDrive/GEOG 761 Group Project/Project Data/raw


## 5. Land Type — ESA WorldCover v200

2021 map, 11 land-cover classes (Sentinel-1/2 derived), exported at its
native 10 m (`WORLDCOVER_SCALE_M`). Matches the architecture diagram's
"Land Type" box directly — no source substitution needed.


In [7]:
land_type_image = fetch_land_cover(AOI)
land_type_path = str(RAW_DIR / f"{AOI_NAME}_land_type.tif")
export_ee_image(land_type_image, land_type_path, AOI, scale=WORLDCOVER_SCALE_M)
print("Saved:", land_type_path)


/usr/local/lib/python3.13/dist-packages/geemap/common.py:12333: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


2021:   0%|          |0/400 tiles [00:00<?]

Saved: /content/drive/MyDrive/GEOG 761 Group Project/Project Data/raw/northland_land_type.tif


## 6. Elevation — LINZ New Zealand LiDAR 1 m DEM

Bare-earth DEM for the "bathtub model" (ground elevation − SAR flood-extent
boundary → water surface elevation → depth). Heights are NZVD2016.

LINZ publishes the national 1 m DEM (LDS layer 121859) as Cloud-Optimised
GeoTIFFs in NZTM (EPSG:2193) in a public S3 bucket, one tile per Topo50 sheet —
no API key needed. Each tile has built-in overviews, so reading it at
`DEM_RES_M` = 10 m only downloads the 8 m overview, not the 1 m data. The tiles
are warped to EPSG:4326 to match every other layer in this notebook. The output
is the AOI's bounding rectangle (nodata over sea and where there's no LiDAR).


In [8]:
elevation_path = str(RAW_DIR / f"{AOI_NAME}_lidar_dem_{DEM_RES_M}m.tif")
fetch_linz_dem(AOI_POLY, elevation_path, res_m=DEM_RES_M)
print("Saved:", elevation_path)


  33 DEM tile(s) intersect the AOI
Saved: /content/drive/MyDrive/GEOG 761 Group Project/Project Data/raw/northland_lidar_dem_10m.tif


## 7. SAR Satellite — Sentinel-1 GRD

Chosen per the group's satellite comparison (free/open access, 10 m
resolution, ~12-day revisit over NZ, used in operational systems like
Copernicus Global Flood Monitoring). The window in section 4 selects the
post-Cyclone Gabrielle ascending pass (14 Feb 2023); expect 2 scenes, the
two adjacent frames of that pass, merged by the median where they overlap.
Exported at its native 10 m (`SAR_SCALE_M`).


In [9]:
# sar_collection = fetch_sar_collection(AOI, START_DATE, END_DATE)
# sar_composite = sar_collection.select(["VV", "VH"]).median().clip(AOI)

# sar_path = str(RAW_DIR / f"{AOI_NAME}_sar_{START_DATE}_to_{END_DATE}.tif")
# export_ee_image(sar_composite, sar_path, AOI, scale=SAR_SCALE_M)
# print("Saved:", sar_path)
# print("Scenes used:", sar_collection.size().getInfo())


## 8. Population Density — WorldPop

Architecture diagram target: WorldPop (NZ, ~100 m) or the Stats NZ Census
meshblock grid. WorldPop is used here since it's published directly as a
GEE ImageCollection; the Stats NZ grid would need a separate download +
join step. 2020 is the latest year in the collection; exported at its native
100 m (`POPULATION_SCALE_M`).


In [10]:
population_image = fetch_population(AOI, WORLDPOP_YEAR)
population_path = str(RAW_DIR / f"{AOI_NAME}_population_{WORLDPOP_YEAR}.tif")
export_ee_image(population_image, population_path, AOI, scale=POPULATION_SCALE_M)
print("Saved:", population_path)


/usr/local/lib/python3.13/dist-packages/geemap/common.py:12333: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/10 tiles [00:00<?]

Saved: /content/drive/MyDrive/GEOG 761 Group Project/Project Data/raw/northland_population_2020.tif


/usr/local/lib/python3.13/dist-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


## 8b. Height above nearest drainage — MERIT Hydro

HAND (`hnd`, metres above the nearest drainage) from MERIT Hydro v1.0.1, exported at its native 90 m. `flood_event_pipeline.ipynb` uses it as a model feature and to define the floodable study area.


In [11]:
HAND_COLLECTION = "MERIT/Hydro/v1_0_1"
HAND_SCALE_M = 90  # MERIT Hydro native resolution

hand_image = ee.Image(HAND_COLLECTION).select("hnd").clip(AOI).rename("hand")
hand_path = str(RAW_DIR / f"{AOI_NAME}_hand.tif")
export_ee_image(hand_image, hand_path, AOI, scale=HAND_SCALE_M)
print("Saved:", hand_path)


MERIT/Hydro/v1_0_1:   0%|          |0/12 tiles [00:00<?]

Saved: /content/drive/MyDrive/GEOG 761 Group Project/Project Data/raw/northland_hand.tif


## 9. Critical Infrastructure — LINZ NZ Building Outlines

Every building footprint in the AOI (LDS layer 101290), fetched through the
LINZ WFS in pages of 10,000. The `use` attribute flags critical buildings
(`Hospital`, `School`, ...) — most are `Unknown`, so the full set doubles as
the building-exposure layer.


In [12]:
buildings = fetch_linz_wfs(101290, AOI_POLY)  # NZ Building Outlines
buildings_path = RAW_DIR / f"{AOI_NAME}_building_outlines.geojson"
buildings.to_file(buildings_path, driver="GeoJSON")
print(f"{len(buildings)} buildings -> {buildings_path}")
print(buildings["use"].value_counts())


160923 buildings -> /content/drive/MyDrive/GEOG 761 Group Project/Project Data/raw/northland_building_outlines.geojson
use
Unknown        159323
School           1461
Hospital           94
Supermarket        45
Name: count, dtype: int64


## 10. Transport Infrastructure — LINZ NZ Road Centrelines

NZ Road Centrelines (Topo, 1:50k, LDS layer 50329), clipped to the AOI.
Includes state highway number, lane count and surface.


In [13]:
roads = fetch_linz_wfs(50329, AOI_POLY)  # NZ Road Centrelines (Topo, 1:50k)
roads_path = RAW_DIR / f"{AOI_NAME}_road_centrelines.geojson"
roads.to_file(roads_path, driver="GeoJSON")
print(f"{len(roads)} road segments -> {roads_path}")


11679 road segments -> /content/drive/MyDrive/GEOG 761 Group Project/Project Data/raw/northland_road_centrelines.geojson


## 11. Summary

All six input layers (plus HAND) for the AOI are now in `data/raw/` (or your mounted
Drive folder if `USE_DRIVE = True`).


In [14]:
for p in sorted(RAW_DIR.glob("*.*")):
    print(p.name)

northland_building_outlines.geojson
northland_hand.tif
northland_land_type.tif
northland_lidar_dem_10m.tif
northland_population_2020.tif
northland_road_centrelines.geojson
